In [1]:
from ultralytics import YOLO
import cv2
import glob
import easyocr
import re


model = YOLO("runs/detect/train-3/weights/best.pt")


reader = easyocr.Reader(['en'], gpu=True)


images = glob.glob("dataset/test/images/*.jpg")

print("IMAGES:", len(images))

def clean(text):
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]', '', text)
    return text

for img_path in images:

    img = cv2.imread(img_path)

    results = model.predict(img, conf=0.25, verbose=False)

    for r in results:
        for box in r.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            crop = img[y1:y2, x1:x2]

            ocr = reader.readtext(crop)

            if ocr:
                text = max(ocr, key=lambda x: x[2])[1]
                text = clean(text)

                print(img_path, "→", text)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


IMAGES: 63


C:\Users\lindo\Downloads\ALPR-RU\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


dataset/test/images\09-19-06-Cam1_jpg.rf.a794b758a807ec6ac08b6d2f82e0f175.jpg → AC
dataset/test/images\09-19-06-Cam2_jpg.rf.2110d0ac45022f2e9062bd320e03c1e1.jpg → X
dataset/test/images\10_jpg.rf.1d44824ffb1234f63f3d2531593524d3.jpg → EO9OBAL16Z
dataset/test/images\10_jpg.rf.1d44824ffb1234f63f3d2531593524d3.jpg → ILE
dataset/test/images\11_6_2014_18_42_24_899_png.rf.8497e3135b77e92d28c28baea2923e5c.jpg → 09991C
dataset/test/images\11_jpg.rf.90f13ef3216a7240666417476e8aca65.jpg → MZOOMME
dataset/test/images\12_6_2014_19_54_57_849_png.rf.5f74991818bfb9bcb5b7ae9755b1b96d.jpg → B118KK35
dataset/test/images\15_jpg.rf.dcc03c3b96729ac3b36e69aa3053a197.jpg → X57SEPI150
dataset/test/images\20230405-112422_jpg.rf.69856ec911d503a58c6ad8db5c6a7537.jpg → O303TEF5
dataset/test/images\22_jpg.rf.0d2db2ce3b52f80d5a111ddc51808342.jpg → X338PM
dataset/test/images\22_jpg.rf.0d2db2ce3b52f80d5a111ddc51808342.jpg → 262
dataset/test/images\34_jpg.rf.8c2bc72a29908b22a9cd0a26d2743c2c.jpg → H048HE
dataset/test/im

In [2]:
import sys
print(sys.executable)

C:\Users\lindo\Downloads\ALPR-RU\venv\Scripts\python.exe


In [2]:
import cv2
import glob
import re
import easyocr
from ultralytics import YOLO


model = YOLO(r"runs/detect/train-3/weights/best.pt")

reader = easyocr.Reader(['en'], gpu=False)

images = glob.glob(r"dataset/test/images/*.jpg")
print("IMAGES:", len(images))


def clean_plate(text):
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]', '', text)

    replacements = {
        "O": "0",
        "I": "1",
        "Z": "2",
        "S": "5",
        "B": "8",
        "G": "6"
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    return text


def is_russian_plate(text):
    return bool(re.match(r'^[A-Z0-9]{6,9}$', text))


for img_path in images:

    img = cv2.imread(img_path)

    results = model.predict(img, conf=0.3, verbose=False)

    for r in results:
        for box in r.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            crop = img[y1:y2, x1:x2]

            ocr_result = reader.readtext(crop)

            text = ""

            if len(ocr_result) > 0:
                ocr_result = sorted(ocr_result, key=lambda x: x[2], reverse=True)
                text = ocr_result[0][1]

            text = clean_plate(text)

            if is_russian_plate(text):
                print(img_path, "→", text)

Using CPU. Note: This module is much faster with a GPU.


IMAGES: 63


C:\Users\lindo\Downloads\ALPR-RU\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


dataset/test/images\11_6_2014_18_42_24_899_png.rf.8497e3135b77e92d28c28baea2923e5c.jpg → 09991C
dataset/test/images\11_jpg.rf.90f13ef3216a7240666417476e8aca65.jpg → M200MME
dataset/test/images\12_6_2014_19_54_57_849_png.rf.5f74991818bfb9bcb5b7ae9755b1b96d.jpg → 8118KK35
dataset/test/images\20230405-112422_jpg.rf.69856ec911d503a58c6ad8db5c6a7537.jpg → 0303TEF5
dataset/test/images\22_jpg.rf.0d2db2ce3b52f80d5a111ddc51808342.jpg → X338PM
dataset/test/images\34_jpg.rf.8c2bc72a29908b22a9cd0a26d2743c2c.jpg → H048HE
dataset/test/images\52_jpg.rf.be9120d87e22941872dd6dac558016f7.jpg → C777AA7E
dataset/test/images\81_jpg.rf.f229b6b499b877a6f420adf20d7ac9f1.jpg → A686MP197
dataset/test/images\87_jpg.rf.8a6e1f842499299fcdcc61e898fc3415.jpg → 0M958M54
dataset/test/images\9_jpg.rf.9715bc1928155c20356f350dade8d172.jpg → 185KX97
dataset/test/images\H087XM-1900_bmp.rf.b8f63839bf4dbab40d525955e9246dba.jpg → H082XN79
dataset/test/images\H130KE-1990_bmp.rf.8b2a555f489bbb21896ff9dce90c12f1.jpg → HT30KE192


In [3]:
import cv2
import re
import glob
import easyocr
from ultralytics import YOLO

# =====================
# MODEL
# =====================
model = YOLO("runs/detect/train-3/weights/best.pt")
reader = easyocr.Reader(['en'], gpu=True)

# =====================
# DATA
# =====================
images = glob.glob("dataset/test/images/*")

# =====================
# CLEANING
# =====================
def clean_plate(text):
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]', '', text)

    replace_map = {
        "O": "0",
        "I": "1",
        "Z": "2",
        "S": "5",
        "B": "8",
        "G": "6"
    }

    for k, v in replace_map.items():
        text = text.replace(k, v)

    return text


def is_russian_plate(text):
    return 6 <= len(text) <= 9


# =====================
# PIPELINE
# =====================
results_list = []

for img_path in images:
    img = cv2.imread(img_path)

    preds = model.predict(img, conf=0.25, verbose=False)

    for r in preds:
        for box in r.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            crop = img[y1:y2, x1:x2]

            ocr_result = reader.readtext(crop)

            text = ""
            if ocr_result:
                text = ocr_result[0][1]

            text = clean_plate(text)

            if is_russian_plate(text):
                print(img_path, "→", text)
                results_list.append((img_path, text))

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
C:\Users\lindo\Downloads\ALPR-RU\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


dataset/test/images\11_jpg.rf.90f13ef3216a7240666417476e8aca65.jpg → M200MME
dataset/test/images\12_6_2014_19_54_57_849_png.rf.5f74991818bfb9bcb5b7ae9755b1b96d.jpg → 8118KK35
dataset/test/images\20230405-112422_jpg.rf.69856ec911d503a58c6ad8db5c6a7537.jpg → 0303TEF5
dataset/test/images\22_jpg.rf.0d2db2ce3b52f80d5a111ddc51808342.jpg → X338PM
dataset/test/images\34_jpg.rf.8c2bc72a29908b22a9cd0a26d2743c2c.jpg → H048HE
dataset/test/images\52_jpg.rf.be9120d87e22941872dd6dac558016f7.jpg → C777AA7E
dataset/test/images\81_jpg.rf.f229b6b499b877a6f420adf20d7ac9f1.jpg → A686MP197
dataset/test/images\87_jpg.rf.8a6e1f842499299fcdcc61e898fc3415.jpg → 0M958M54
dataset/test/images\88_jpg.rf.9f805e1578933874a29fea0b530c6b28.jpg → T552YH
dataset/test/images\99_png.rf.c13d53f3007b60f1d43d9613160ba72a.jpg → K707YY109
dataset/test/images\9_jpg.rf.9715bc1928155c20356f350dade8d172.jpg → 185KX97
dataset/test/images\H087XM-1900_bmp.rf.b8f63839bf4dbab40d525955e9246dba.jpg → H082XN79
dataset/test/images\H097AH-77